# Homework 08: Exploratory Data Analysis

Full profiling, distribution and relationship plots, a time-series read, and a reusable
`eda_summary()` helper on a synthetic customer-transactions dataset.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

sns.set_theme(context='talk', style='whitegrid')
np.random.seed(8)
pd.set_option('display.max_columns', 100)

n = 160
df = pd.DataFrame({
    'date': pd.date_range('2021-02-01', periods=n, freq='D'),
    'region': np.random.choice(['North', 'South', 'East', 'West'], size=n),
    'age': np.random.normal(40, 8, size=n).clip(22, 70).round(1),
    'income': np.random.lognormal(mean=10.6, sigma=0.3, size=n).round(2),
    'transactions': np.random.poisson(lam=3, size=n),
})
base = df['income'] * 0.0015 + df['transactions'] * 18 + np.random.normal(0, 40, size=n)
df['spend'] = np.maximum(0, base).round(2)

df.loc[np.random.choice(df.index, 5, replace=False), 'income'] = np.nan
df.loc[np.random.choice(df.index, 3, replace=False), 'spend'] = np.nan
df.loc[np.random.choice(df.index, 2, replace=False), 'transactions'] = df['transactions'].max() + 12
df.head()

,date,region,age,income,transactions,spend
0,2021-02-01,West,37.6,28086.81,4,73.35
1,2021-02-02,North,43.0,33034.75,1,52.37
2,2021-02-03,South,38.2,50045.39,2,131.85
3,2021-02-04,South,24.9,39467.28,4,147.58
4,2021-02-05,South,59.8,31201.65,1,86.76


## 1. First Look

In [2]:
df.info()
print()
print('Missing values per column:')
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          160 non-null    datetime64[us]
 1   region        160 non-null    str           
 2   age           160 non-null    float64       
 3   income        155 non-null    float64       
 4   transactions  160 non-null    int32         
 5   spend         157 non-null    float64       
dtypes: datetime64[us](1), float64(3), int32(1), str(1)
memory usage: 7.7 KB

Missing values per column:
date            0
region          0
age             0
income          5
transactions    0
spend           3
dtype: int64


## 2. Numeric Profile

In [3]:
numeric_cols = ['age', 'income', 'transactions', 'spend']
desc = df[numeric_cols].describe().T
desc['skew'] = [skew(df[c].dropna()) for c in desc.index]
desc['kurtosis'] = [kurtosis(df[c].dropna()) for c in desc.index]
desc

,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
age,160.0,40.018750,8.458676,22.00,34.70,40.15,44.925,61.10,0.069538,-0.080125
income,155.0,41983.866323,13262.457038,17928.80,32471.53,39332.52,49697.690,87052.40,0.993336,0.918722
transactions,160.0,3.237500,2.585610,0.00,2.00,3.00,4.000,20.00,3.466078,19.984802
spend,157.0,117.291592,51.768645,0.54,77.25,119.32,153.340,280.05,0.130860,-0.084917


## 2b. Categorical Profile

`.describe()` above only covers numeric columns. `region` is the one non-numeric column here.
Counts alone hide how lopsided a split is, so this shows both counts and proportions.

In [4]:
cat_cols = df.select_dtypes(exclude='number').columns.drop('date')
for col in cat_cols:
    counts = df[col].value_counts()
    props = df[col].value_counts(normalize=True)
    print(f'--- {col} ---')
    print(pd.DataFrame({'count': counts, 'proportion': props.round(3)}))
    print()

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(data=df, x='region', ax=ax)
ax.set_title('Region counts')
fig.tight_layout()
fig.savefig('region_counts.png', dpi=110)
plt.close(fig)

--- region ---
        count  proportion
region                   
West       47       0.294
East       42       0.262
South      36       0.225
North      35       0.219



## 3. Distributions

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df['income'], kde=True, ax=axes[0])
axes[0].set_title('Income distribution')
sns.boxplot(x=df['transactions'], ax=axes[1])
axes[1].set_title('Transactions (outliers)')
sns.histplot(df['spend'], kde=True, ax=axes[2])
axes[2].set_title('Spend distribution')
fig.tight_layout()
fig.savefig('distributions.png', dpi=110)
plt.close(fig)
print('Saved distributions.png')

Saved distributions.png


## 4. Relationships

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.scatterplot(data=df, x='income', y='spend', hue='region', ax=axes[0])
axes[0].set_title('Income vs spend')
sns.scatterplot(data=df, x='age', y='spend', ax=axes[1])
axes[1].set_title('Age vs spend')
fig.tight_layout()
fig.savefig('relationships.png', dpi=110)
plt.close(fig)
print('Saved relationships.png')

Saved relationships.png

## 4b. Over Time

Sorted by date first, since a rolling read over out-of-order rows means nothing. Also checking
for gaps in the date index before drawing any conclusion about trend or seasonality.

In [7]:
df_ts = df.sort_values('date').set_index('date')

full_range = pd.date_range(df_ts.index.min(), df_ts.index.max(), freq='D')
missing_dates = full_range.difference(df_ts.index)
print('Date range:', df_ts.index.min().date(), 'to', df_ts.index.max().date())
print('Missing dates in range:', len(missing_dates))

fig, ax = plt.subplots(figsize=(10, 4))
df_ts['spend'].plot(ax=ax, label='daily spend', alpha=0.5)
df_ts['spend'].rolling(7).mean().plot(ax=ax, label='7-day rolling mean', linewidth=2)
ax.set_title('Spend over time')
ax.legend()
fig.tight_layout()
fig.savefig('spend_over_time.png', dpi=110)
plt.close(fig)
print('Saved spend_over_time.png')

Date range: 2021-02-01 to 2021-07-10
Missing dates in range: 0
Saved spend_over_time.png


What the time-series plot shows: no missing dates in range, so the daily index is complete
and a rolling window means what it should. The 7-day rolling mean stays close to flat across
the 160 days, no visible trend, no obvious weekly seasonality, no level shift. That reads as
genuinely nothing, which is itself a finding. This synthetic spend series was generated as
independent noise per day with no time structure baked in. For stage 09 and 10b, that means
lag and rolling features on this particular column would mostly encode noise rather than real
signal, worth checking directly with an autocorrelation plot before trusting them.

## 5. Correlation Matrix (stretch)

In [8]:
corr = df[numeric_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlation matrix')
fig.tight_layout()
fig.savefig('correlation_matrix.png', dpi=110)
plt.close(fig)
corr

,age,income,transactions,spend
age,1.000000,-0.123160,0.037754,0.008174
income,-0.123160,1.000000,0.063573,0.307307
transactions,0.037754,0.063573,1.000000,0.480685
spend,0.008174,0.307307,0.480685,1.000000


income and transactions both correlate with spend, by construction spend was built from
both, and income and transactions barely correlate with each other. For modeling, both are
worth keeping as separate features rather than picking one, since neither substitutes for
the other.

## 5b. Reusable Helper: `src/eda.py`

`eda_summary(df)` from the lecture, extended to flag columns worth attention before stage 09:
high missingness, near-zero variance, or one category dominating.

In [9]:
from src.eda import eda_summary

summary = eda_summary(df, numeric_cols=numeric_cols)
print('shape:', summary['shape'])
print('missing:', summary['missing'])
print('flags:', summary['flags'] if summary['flags'] else 'none')
summary['numeric_profile']

shape: (160, 6)
missing: {'date': 0, 'region': 0, 'age': 0, 'income': 5, 'transactions': 0, 'spend': 3}
flags: none


,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
age,160.0,40.018750,8.458676,22.00,34.70,40.15,44.925,61.10,0.069538,-0.080125
income,155.0,41983.866323,13262.457038,17928.80,32471.53,39332.52,49697.690,87052.40,0.993336,0.918722
transactions,160.0,3.237500,2.585610,0.00,2.00,3.00,4.000,20.00,3.466078,19.984802
spend,157.0,117.291592,51.768645,0.54,77.25,119.32,153.340,280.05,0.130860,-0.084917


## 6. Insights and Assumptions

Top 3 insights:

1. income is right-skewed, since it was generated log-normal, so its mean overstates a
   typical customer. Median is the more honest summary, and a log transform is worth trying
   before feeding it to a linear model.
2. The two injected outlier rows in transactions show up clearly in the boxplot and would
   have inflated a naive mean. eda_summary's flags do not catch this one, since two rows out
   of 160 is not enough to trip the near-zero-variance or missingness checks. That is a real
   gap, outlier flags need a dedicated check, which is exactly what stage 07's functions are for.
3. spend over time is flat, with no trend or seasonality, so any lag or rolling feature built
   on it in stage 09 should be validated against an autocorrelation check first rather than
   assumed useful just because the data has a date column.

Assumptions and risks:

- The synthetic generator assumes spend is a noisy linear function of income and
  transactions. A real dataset will not be this clean, and the actual relationship could be
  nonlinear or have interaction effects this EDA would not surface without more targeted plots.
- region shows no strong imbalance in this run, checked in the categorical profile above, but
  that is a property of this particular random seed, not something to assume holds for a
  different dataset.

Implications for next step: address the income skew and the two transactions outliers before
feature engineering, and treat spend's lack of time structure as a reason to test rather than
assume any lag features actually help in stage 09.